# BP7 Gate 2 — Data Verification & Feature/Taxonomy Engineering
**Customer360 Navigator Enterprise Suite — Customer360 Navigator Decision Engine**

## Why this notebook exists, and the real design gap it resolves

BP7 Gate 1's own real, live-run `policy.json` (`target_definition.known_dependency_gaps`) found
that BP1/BP2/BP3's real `gate5_decision_records.csv` artifacts are each that BP's own independent
held-out TEST-SPLIT predictions, keyed only by a `row_index` local to that BP's own split -
live-verified there to carry no `Complaint ID` column. Three independently-split BPs cannot be
joined row-for-row from those artifacts alone. Gate 1 named two possible resolutions and deferred
the choice to Gate 2/3. **This notebook implements the second one: a BP7-owned common re-scoring
pass** - never retroactively touching BP1/BP2/BP3's own already-real-run-confirmed Gate 3/4/5
notebooks, champions, hyperparameters, or thresholds. It reuses each BP's real persisted champion
bundle (`src/models/model_persistence.py`, already delivered, already tested) purely for
inference, re-applied to the FULL real CFPB population, keyed by the real, always-present
`Complaint ID` column - never a join across three disjoint test splits.

## A real finding this notebook's design had to resolve honestly: BP1 is NOT re-scored here

BP1's own real, live-run Gate 1 `policy.json` states plainly that "no CFPB row-level data is ever
joined into BANKING77 training or evaluation data" and that "BP1's trainable text classifier is
built and evaluated on BANKING77 alone, because the CFPB extract used in this project has no
narrative/complaint-text column." `model_persistence.predict_bp1()` takes raw `texts` - and the
real, live-re-confirmed 15-column CFPB extract has no free-text field to hand it. Synthesizing
pseudo-text from structured fields would feed BP1's classifier an input distribution it was never
fit or evaluated on and present whatever it returns as a real prediction - exactly the
fabricated-result-for-a-missing-input pattern this project's zero-fabrication rule (and BP1 Gate
1's own leakage_rules) forbid. BP1's real, already-computed, CFPB-row-level contribution used here
instead is the taxonomy CROSSWALK BP1 Gate 2 already built and persisted
(`data/processed/cfpb_common_taxonomy_gold.parquet`, `common_taxonomy_bucket`) - carried forward as
optional context only, exactly as BP7 Gate 1's policy.json scopes BP1
(`OPTIONAL_CONTEXT_ONLY`). See `src/features/bp7_decision_engine_features.py`'s own module
docstring for the full reasoning.

BP2 and BP3 ARE re-scored via `model_persistence.predict_bp2()`/`predict_bp3()`, because both
operate on real structured CFPB columns present on every real row - a valid, real input for every
row, never a fabricated one. Each BP's own real, already-delivered null-handling convention is
reused exactly as that BP's own Gate 3 notebook defines it (verified live against that notebook's
own delivered source before this module was written - BP2 fills categorical nulls with the
literal `"MISSING"`; BP3 uses its own per-column `NULL_SENTINEL_MAP`, reused unmodified).

## Zero-fabrication bundle-availability handling

If a BP's real persisted joblib bundle is not present on disk (or fails to load), that BP's
contribution is reported as `NOT_AVAILABLE_BUNDLE_NOT_YET_PERSISTED` for every row - never a
mock/fabricated prediction, matching `src/services/bp4_decision_service.py`'s own established
503-not-a-fake-result idiom (itself built on `services.service_common.ModelBundleHandle`, reused
unmodified here). This notebook still completes successfully and writes whatever real, available
combination of upstream contributions it can, rather than failing the whole gate over one missing
upstream bundle.

## BP4 (cluster-level join) and BP5 (outcome-level qualitative context)

BP4 fits no model - its real Gate 5 output (`gate5_cluster_decision_report.csv`) is already at the
issue-cluster grain BP7 Gate 1's policy.json names, joined onto complaint-level rows here via the
real, always-present `(Company, Product, Sub-product, Issue, Sub-issue)` key (reused unmodified
from `features.bp4_journey_features.CLUSTER_KEY`). An unmatched row's BP4 fields (only those
fields) are reported `UNSCORED_MISSING_UPSTREAM_INPUT`, never the whole row dropped.

BP5 Gate 5's own real output is an OUTCOME-level "prioritized root-cause report," not a
per-complaint prediction - live-reconfirmed here (again) to carry no per-row key, so no row-level
join is attempted. Instead, this notebook builds a small, real, structurally-grounded lookup from
BP5's own real `category_level_findings_by_field` top-K association categories and flags, for
every complaint row, whether its own real Product/Sub-product/Issue/Sub-issue/Submitted via value
is one of BP5's own real reported top-K categories for each of BP5's two real outcome fields -
qualitative context only, never a numeric weight, never a causal claim (BP5's own
`association_not_causation_disclaimer`, carried forward verbatim).

## Standing rules this notebook follows

- **WARP**: Polars lazy scans for the real CFPB file, category dtypes reused from
  `taxonomy_mapper.CFPB_DTYPES` (not re-derived), batched inference (never one dense array for the
  full ~1.05M-row population at once).
- **Zero-fabrication / live drift checks**: Complaint ID uniqueness, the barred-column list, and
  the demographic-adjacent `Tags` values are all re-checked live against the real CFPB file in this
  notebook's own Section 5/6 - never trusted from any prior BP's own finding without re-measuring.
- **Barred inputs**: `Company response to consumer`, `Timely response?`, `Date received`,
  `Date sent to company`, and `Tags` are never selected into this notebook's working feature frame
  at any point (structurally verified in Section 6, not merely asserted in prose) - matching BP7
  Gate 1's own real `leakage_rules` verbatim.
- **Execution boundary**: this notebook is delivered as source only. It has never been executed by
  Claude - only the user runs real pipeline notebooks, per this project's standing rule. Every
  design choice above was verified in a disposable sandbox against a synthetic fixture mirroring
  the exact real schemas confirmed live on the real device (never against real customer data, and
  no sandbox output is delivered) as a pre-delivery check, not a substitute for the user's own real
  run.

## Prerequisites

- BP7 Gate 1 must be real-run at least once (`configs/bp7_customer_navigator_decision_engine.yaml`
  front matter and `notebooks/bp7_customer_navigator_decision_engine/artifacts/policy.json` must
  both exist) - this notebook raises immediately if either is missing.
- For BP2/BP3 to be re-scored (rather than reported `NOT_AVAILABLE_BUNDLE_NOT_YET_PERSISTED`), run
  `bp2_customer_friction_classification_model_persistence.ipynb` and
  `bp3_complaint_escalation_prediction_model_persistence.ipynb` for real first, if not already
  done. This notebook still completes successfully either way - it never fails the whole gate over
  one missing upstream bundle.

## What this notebook does NOT do

- No `priority_score`, `intervention_flag`, or `recommended_action` is computed here - Gate 1's own
  policy.json explicitly scopes the deterministic weighted-rule combination (and its weights/
  thresholds) to Gate 3/4, never invented ahead of schedule here.
- No upstream BP's own champion, hyperparameters, or threshold is retrained, refit, or changed -
  every BP2/BP3 prediction here is pure inference through that BP's own already-validated,
  already-persisted pipeline.
- No causal claim is made anywhere in this notebook about BP5's context fields - association only,
  per BP5's own `association_not_causation_disclaimer`, carried forward verbatim.


In [ ]:
"""
Customer360 Navigator Enterprise Suite - BP7 Gate 2 data verification / re-scoring / join
notebook. Single consolidated code cell (platform convention). Idempotent - safe to re-run.
"""

import os, sys, json, warnings
from datetime import datetime, timezone
from pathlib import Path

warnings.filterwarnings("ignore")


# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
def _find_project_root() -> Path:
    marker = "PROJECT_STRUCTURE_LOCKED.md"
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        if (Path(env_override) / marker).exists():
            return Path(env_override)
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {env_override!r} but {marker} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    cur = start
    for _ in range(8):
        if (cur / marker).exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker in filenames:
            return Path(depth_root)

    raise RuntimeError(
        f"Could not resolve PROJECT_ROOT: no {marker} found by walking up from {start}, nor by "
        "searching up to 3 levels below it. Fix: add a cell at the TOP of this notebook (before "
        "this cell runs) with:\n"
        '    import os; os.environ["C360_PROJECT_ROOT"] = r"C:\\Users\\rnand\\Documents\\'
        'Customer360_Navigator_Enterprise_Suite"\n'
        "then re-run from the top."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP performance configuration - FIRST, before any heavy import
# ============================================================
from utils.performance_setup import configure_performance, memory_headroom_gb

perf_summary = configure_performance(project_root=PROJECT_ROOT, verbose=True)
print(f"[WARP] Headroom before heavy work: {memory_headroom_gb()} GB")

# ============================================================
# SECTION 3: Heavy imports (only after WARP configuration)
# ============================================================
import polars as pl
import yaml
from IPython.display import display

from taxonomy.taxonomy_mapper import CFPB_DTYPES
from utils.bp1_config_sync import write_gate_block  # noqa: E402
from features.bp7_decision_engine_features import (  # noqa: E402
    BARRED_COLUMNS,
    BP1_OUT_OF_SCOPE_BUCKET_LITERAL,
    UNSCORED_MISSING_UPSTREAM_INPUT,
    NOT_AVAILABLE_BUNDLE_NOT_YET_PERSISTED,
    load_bp2_bp3_bundles,
    attach_bp1_taxonomy_context,
    score_bp2_population,
    score_bp3_population,
    join_bp4_cluster_tier,
    build_bp5_context_lookup,
    attach_bp5_context,
    feature_lineage_table,
    coverage_report,
)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
CONFIGS_DIR = PROJECT_ROOT / "configs"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp7_customer_navigator_decision_engine" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

CFPB_PATH = DATA_RAW / "cfpb_complaints.csv"
GOLD_TAXONOMY_PATH = DATA_PROCESSED / "cfpb_common_taxonomy_gold.parquet"
BP4_CSV_PATH = (
    PROJECT_ROOT
    / "notebooks"
    / "bp4_customer_journey_analytics"
    / "artifacts"
    / "gate5_cluster_decision_report.csv"
)
BP5_OUTCOME1_JSON = (
    PROJECT_ROOT
    / "notebooks"
    / "bp5_root_cause_driver_analytics"
    / "artifacts"
    / "gate5_prioritized_root_cause_report_outcome_1_intervention_required.json"
)
BP5_OUTCOME2_JSON = (
    PROJECT_ROOT
    / "notebooks"
    / "bp5_root_cause_driver_analytics"
    / "artifacts"
    / "gate5_prioritized_root_cause_report_outcome_2_timely_response_failure.json"
)
GOLD_OUT_PATH = DATA_PROCESSED / "cfpb_decision_engine_context_gold.parquet"
BP7_CONFIG_PATH = CONFIGS_DIR / "bp7_customer_navigator_decision_engine.yaml"
POLICY_JSON_PATH = ARTIFACTS_DIR / "policy.json"
LINEAGE_PATH = ARTIFACTS_DIR / "gate2_feature_lineage.csv"
SUMMARY_JSON_PATH = ARTIFACTS_DIR / "gate2_rescoring_summary.json"

for p in (CFPB_PATH, BP7_CONFIG_PATH, POLICY_JSON_PATH):
    if not p.exists():
        raise FileNotFoundError(
            f"Required input not found: {p}. Confirm BP7 Gate 1 has been real-run at least once "
            "(the config front matter and policy.json artifact are both prerequisites)."
        )

with open(POLICY_JSON_PATH, "r", encoding="utf-8") as f:
    gate1_policy = json.load(f)
print(f"[OK] Loaded Gate 1 policy.json (generated_at_utc={gate1_policy['generated_at_utc']}).")

with open(BP7_CONFIG_PATH, "r", encoding="utf-8") as f:
    _front_matter_only = yaml.safe_load(f.read().split("# --- Gate", 1)[0])
gate1_status_confirmed = "gate1_confirmed" in str(_front_matter_only.get("status", ""))
assert gate1_status_confirmed, (
    f"configs/bp7_customer_navigator_decision_engine.yaml status is "
    f"'{_front_matter_only.get('status')}' - expected it to contain 'gate1_confirmed'. Run Gate 1 "
    "for real before Gate 2."
)
print(f"[OK] Gate 1 prerequisite confirmed: status='{_front_matter_only.get('status')}'")

# ============================================================
# SECTION 4: LIVE re-check - Complaint ID uniqueness/non-null (BP7 Gate 1's own re-scoring
# resolution depends structurally on this real key being unique) and the real row count, never
# assumed from Gate 1's own baked-in figure.
# ============================================================
cfpb_lazy_full = pl.scan_csv(CFPB_PATH, schema_overrides=CFPB_DTYPES)
live_row_count = cfpb_lazy_full.select(pl.len()).collect().item()
live_complaint_id_stats = cfpb_lazy_full.select(
    pl.col("Complaint ID").n_unique().alias("n_unique"),
    pl.col("Complaint ID").null_count().alias("n_null"),
).collect()
n_unique_complaint_id = int(live_complaint_id_stats["n_unique"][0])
n_null_complaint_id = int(live_complaint_id_stats["n_null"][0])
complaint_id_is_unique_and_nonnull = n_unique_complaint_id == live_row_count and n_null_complaint_id == 0
print(
    f"[LIVE CHECK] Complaint ID: {n_unique_complaint_id:,} unique / {live_row_count:,} rows, "
    f"{n_null_complaint_id} null -> unique_and_nonnull={complaint_id_is_unique_and_nonnull}"
)
print(
    f"[INFO] Live CFPB row count: {live_row_count:,} (Gate 1's own recorded figure: "
    f"{gate1_policy['live_checks']['cfpb_row_count']:,}) - informational only; Gate 2's own exit "
    "criteria below do not require these to match a fixture/test run, only a real run against the "
    "same real file Gate 1 saw."
)

# ============================================================
# SECTION 5: LIVE re-check of the demographic-adjacent Tags values (BP7 Gate 1's own leakage_rules
# barred-column re-check convention, re-verified again here rather than trusted from Gate 1 alone).
# ============================================================
live_tags_values = set(
    cfpb_lazy_full.select(pl.col("Tags").unique()).collect()["Tags"].drop_nulls().to_list()
)
KNOWN_DEMOGRAPHIC_ADJACENT_TAGS = {"Servicemember", "Older American", "Older American, Servicemember"}
unexpected_tag_values = live_tags_values - KNOWN_DEMOGRAPHIC_ADJACENT_TAGS
print(f"[LIVE CHECK] Real non-null Tags values found: {sorted(live_tags_values)}")
if unexpected_tag_values:
    print(
        f"[DRIFT DETECTED] Unexpected Tags value(s) not seen at prior BPs' own Gate 1: {sorted(unexpected_tag_values)}"
    )
else:
    print(
        "[OK] Every real non-null Tags value is one of the known demographic-adjacent values - Tags stays barred."
    )

# ============================================================
# SECTION 6: Build the working feature frame - structurally excludes every barred column by
# construction (a .select() naming only the allowed columns, not an after-the-fact drop).
# ============================================================
ALLOWED_BASE_COLUMNS = [
    "Complaint ID",
    "Product",
    "Sub-product",
    "Issue",
    "Sub-issue",
    "State",
    "Submitted via",
    "Company",
]
assert not any(c in ALLOWED_BASE_COLUMNS for c in BARRED_COLUMNS), (
    "A barred column leaked into ALLOWED_BASE_COLUMNS - see BARRED_COLUMNS in "
    "src/features/bp7_decision_engine_features.py."
)
cfpb_lazy_working = cfpb_lazy_full.select(ALLOWED_BASE_COLUMNS)
print(f"[OK] Working feature frame columns (barred columns structurally excluded): {ALLOWED_BASE_COLUMNS}")

# ============================================================
# SECTION 7: BP1 - real taxonomy-crosswalk context only (never predict_bp1() on CFPB - see this
# notebook's own markdown cell for the full reasoning).
# ============================================================
cfpb_lazy_working = attach_bp1_taxonomy_context(cfpb_lazy_working, GOLD_TAXONOMY_PATH)
base_pl = cfpb_lazy_working.collect()
print(f"[OK] Base frame collected: {base_pl.height:,} rows x {base_pl.width} columns.")
bp1_context_counts = base_pl["bp1_context_status"].value_counts().sort("bp1_context_status")
print("\n=== BP1 taxonomy-context status (real, live) ===")
display(bp1_context_counts.to_pandas())

bp1_gold_present = GOLD_TAXONOMY_PATH.exists()
if bp1_gold_present:
    live_out_of_scope_literal_found = bool(
        (base_pl["common_taxonomy_bucket"] == BP1_OUT_OF_SCOPE_BUCKET_LITERAL).any()
    )
    print(
        f"[LIVE CHECK] BP1's real out-of-scope sentinel literal {BP1_OUT_OF_SCOPE_BUCKET_LITERAL!r} "
        f"found in the real Gold taxonomy layer: {live_out_of_scope_literal_found}"
    )
else:
    print(
        f"[SKIPPED] {GOLD_TAXONOMY_PATH.relative_to(PROJECT_ROOT)} not present - every row reported "
        f"{NOT_AVAILABLE_BUNDLE_NOT_YET_PERSISTED} for bp1_context_status."
    )

# ============================================================
# SECTION 8: Load BP2/BP3's real persisted bundles - honestly reports which are available for
# THIS real run, never fabricates a missing one (mirrors src/services/bp4_decision_service.py's
# own 503-not-a-fake-result idiom, built on services.service_common.ModelBundleHandle).
# ============================================================
bundles = load_bp2_bp3_bundles(PROJECT_ROOT)
for bp_id, handle in bundles.items():
    status = "LOADED" if handle.is_loaded else f"NOT LOADED ({handle.error})"
    print(f"[BUNDLE STATUS] {bp_id}: {status}")

# ============================================================
# SECTION 9: BP2 - re-score the full real population if (and only if) the real bundle loaded.
# ============================================================
if bundles["bp2"].is_loaded:
    bp2_scores_pd = score_bp2_population(bundles["bp2"], base_pl)
    bp2_scores_pl = pl.from_pandas(bp2_scores_pd)
    base_pl = base_pl.join(bp2_scores_pl, on="Complaint ID", how="left")
    print(f"[OK] BP2 re-scored {bp2_scores_pl.height:,} real rows.")
else:
    base_pl = base_pl.with_columns(
        pl.lit(None, dtype=pl.Utf8).alias("bp2_predicted_label"),
        pl.lit(None, dtype=pl.Float64).alias("bp2_confidence"),
    )
    print(
        f"[SKIPPED] BP2 bundle not available - every row reported {NOT_AVAILABLE_BUNDLE_NOT_YET_PERSISTED}."
    )

# ============================================================
# SECTION 10: BP3 - re-score the full real population if (and only if) the real bundle loaded.
# ============================================================
if bundles["bp3"].is_loaded:
    bp3_scores_pd = score_bp3_population(bundles["bp3"], base_pl)
    bp3_scores_pl = pl.from_pandas(bp3_scores_pd)
    base_pl = base_pl.join(bp3_scores_pl, on="Complaint ID", how="left")
    print(f"[OK] BP3 re-scored {bp3_scores_pl.height:,} real rows.")
else:
    base_pl = base_pl.with_columns(
        pl.lit(None, dtype=pl.Utf8).alias("bp3_predicted_label"),
        pl.lit(None, dtype=pl.Float64).alias("bp3_probability_positive_class"),
    )
    print(
        f"[SKIPPED] BP3 bundle not available - every row reported {NOT_AVAILABLE_BUNDLE_NOT_YET_PERSISTED}."
    )

# ============================================================
# SECTION 11: BP4 - real cluster-level Gate 5 output, left-joined via the real CLUSTER_KEY.
# ============================================================
base_pl = join_bp4_cluster_tier(base_pl.lazy(), BP4_CSV_PATH).collect()
bp4_join_counts = base_pl["bp4_join_status"].value_counts().sort("bp4_join_status")
print("\n=== BP4 cluster-join status (real, live) ===")
display(bp4_join_counts.to_pandas())

# ============================================================
# SECTION 12: BP5 - real, outcome-level qualitative association context (no row-level join
# attempted - structurally impossible, re-confirmed live: neither real Gate 5 artifact carries a
# per-row key).
# ============================================================
bp5_lookup = build_bp5_context_lookup(BP5_OUTCOME1_JSON, BP5_OUTCOME2_JSON)
base_pl = attach_bp5_context(base_pl.lazy(), bp5_lookup).collect()
for outcome_idx in (1, 2):
    col = f"bp5_outcome_{outcome_idx}_context"
    print(f"\n=== BP5 {col} (real, live) ===")
    display(base_pl[col].value_counts().sort(col).to_pandas())

# ============================================================
# SECTION 13: Real per-field coverage statistics (Gate 2's own 'zero nulls silently dropped'
# disclosure convention).
# ============================================================
coverage = coverage_report(base_pl)
print("\n=== REAL PER-FIELD COVERAGE (BP7 Gate 2) ===")
print(json.dumps(coverage, indent=2, default=str))

# ============================================================
# SECTION 14: Write the real Gold-layer context artifact (Parquet, WARP) - Complaint-ID-keyed,
# carrying every real upstream contribution this gate could compute, plus its own honest
# availability/status columns. No priority_score/intervention_flag/recommended_action here -
# Gate 1's own policy.json scopes those to Gate 3/4.
# ============================================================
base_pl.write_parquet(GOLD_OUT_PATH)
print(f"\n[SAVED] {GOLD_OUT_PATH.relative_to(PROJECT_ROOT)} ({base_pl.height:,} rows x {base_pl.width} cols)")

# ============================================================
# SECTION 15: Write the feature-lineage table (Gate 2's own named exit-criterion artifact).
# ============================================================
lineage = feature_lineage_table()
lineage.write_csv(LINEAGE_PATH)
print(f"[SAVED] {LINEAGE_PATH.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 16: Write the Gate 2 JSON summary artifact.
# ============================================================
gate2_summary = {
    "bp_id": "bp7",
    "gate": 2,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "live_row_count": live_row_count,
    "gate1_recorded_row_count": gate1_policy["live_checks"]["cfpb_row_count"],
    "complaint_id_is_unique_and_nonnull": complaint_id_is_unique_and_nonnull,
    "barred_columns_structurally_excluded": BARRED_COLUMNS,
    "bundle_availability": {bp_id: handle.is_loaded for bp_id, handle in bundles.items()},
    "bp5_gate5_artifacts_available": bp5_lookup is not None,
    "coverage": coverage,
    "gold_layer_path": str(GOLD_OUT_PATH.relative_to(PROJECT_ROOT).as_posix()),
    "gold_layer_rows_written": base_pl.height,
    "feature_lineage_path": str(LINEAGE_PATH.relative_to(PROJECT_ROOT).as_posix()),
}
with open(SUMMARY_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(gate2_summary, f, indent=2, default=str)
print(f"[SAVED] {SUMMARY_JSON_PATH.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 17: Write the Gate 2 config block (marker-based, order-independent - reuses
# bp1_config_sync.py unmodified, per BP7 Gate 1's own assumptions).
# ============================================================
gate2_marker = "# --- Gate 2 (Data Verification & Feature/Taxonomy Engineering) results (appended, idempotent overwrite) ---"
gate2_block_lines = [
    f"live_row_count: {live_row_count}",
    f"complaint_id_is_unique_and_nonnull: {complaint_id_is_unique_and_nonnull}",
    f"bp2_bundle_available: {bundles['bp2'].is_loaded}",
    f"bp3_bundle_available: {bundles['bp3'].is_loaded}",
    f"bp5_gate5_artifacts_available: {bp5_lookup is not None}",
    f'gold_layer_path: "{GOLD_OUT_PATH.relative_to(PROJECT_ROOT).as_posix()}"',
    f"gold_layer_rows_written: {base_pl.height}",
    f'feature_lineage_path: "{LINEAGE_PATH.relative_to(PROJECT_ROOT).as_posix()}"',
    f'gate2_summary_path: "{SUMMARY_JSON_PATH.relative_to(PROJECT_ROOT).as_posix()}"',
]
write_gate_block(BP7_CONFIG_PATH, gate2_marker, gate2_block_lines)
print(f"[SAVED] gate2 block written to {BP7_CONFIG_PATH.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 18: Structural integrity checks - raise AssertionError, never silently pass.
# ============================================================
bp4_join_status_values = set(base_pl["bp4_join_status"].unique().to_list())
bp4_join_status_values_known = bp4_join_status_values <= {
    "BP4_JOINED",
    UNSCORED_MISSING_UPSTREAM_INPUT,
    NOT_AVAILABLE_BUNDLE_NOT_YET_PERSISTED,
}

checks = {
    "complaint_id_is_unique_and_nonnull": complaint_id_is_unique_and_nonnull,
    "no_unexpected_tags_value_found": not unexpected_tag_values,
    "no_barred_column_in_working_frame": not any(c in base_pl.columns for c in BARRED_COLUMNS),
    "gold_layer_row_count_equals_live_row_count": base_pl.height == live_row_count,
    "bp4_join_status_values_all_known_sentinels": bp4_join_status_values_known,
    "bp1_context_status_present_for_every_row": base_pl["bp1_context_status"].null_count() == 0,
    "bp4_join_status_present_for_every_row": base_pl["bp4_join_status"].null_count() == 0,
    "bp5_outcome_1_context_present_for_every_row": base_pl["bp5_outcome_1_context"].null_count() == 0,
    "bp5_outcome_2_context_present_for_every_row": base_pl["bp5_outcome_2_context"].null_count() == 0,
    "feature_lineage_csv_written": LINEAGE_PATH.exists(),
    "gate2_summary_json_written": SUMMARY_JSON_PATH.exists(),
    "gold_layer_parquet_written": GOLD_OUT_PATH.exists(),
    "bp7_config_gate2_block_written": BP7_CONFIG_PATH.exists(),
}

print("\n=== INTEGRITY CHECKS ===")
for name, passed in checks.items():
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {name}")
    assert passed, f"[CHECK FAILED] {name}"

print(
    "\n[ALL CHECKS PASSED] BP7 Gate 2 complete - every real row tagged with its own honest "
    f"upstream-availability status across BP1 (taxonomy context), BP2 (bundle_available="
    f"{bundles['bp2'].is_loaded}), BP3 (bundle_available={bundles['bp3'].is_loaded}), BP4 (real "
    f"cluster join), and BP5 (real outcome-level association context, artifacts_available="
    f"{bp5_lookup is not None}) - none dropped, none fabricated. No priority_score/"
    "intervention_flag/recommended_action computed here (Gate 1's own policy.json scopes the "
    "deterministic weighted rule to Gate 3/4). Proceed to BP7 Gate 3 next."
)
